In [1]:
# === BACKEND SETUP (qiskit_backend_config) ===
# -----------------------------------------------
# Common backend helper — works with Qiskit 2.x / qiskit-aer / qiskit-ibm-runtime.
# Provides drop-in replacements for the removed Aer, BasicAer, execute(), and IBMQ.
#
# HOW TO SELECT A BACKEND
# -----------------------
# Simulator (default, runs locally — no credentials needed):
#   BACKEND_NAME = "qasm_simulator"        # shot-based measurements
#   BACKEND_NAME = "statevector_simulator" # full statevector
#   BACKEND_NAME = "unitary_simulator"     # full unitary matrix
#
# Real IBM hardware (requires IBM Quantum account):
#   Set the environment variable:  IBM_QUANTUM_TOKEN=<your_token>
#   Then choose a device, e.g.:
#   BACKEND_NAME = "ibm_brisbane"
#   BACKEND_NAME = "least_busy"  # auto-pick least-busy device
#
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
from qiskit_backend_config import (get_backend, execute, Aer, BasicAer,
                                   IBMQ, least_busy, job_monitor, assemble, combine)

# ── Choose your backend here ──────────────────────────────────────────────
BACKEND_NAME = "qasm_simulator"  # change to a device name for real hardware
# ── (Optional) IBM Quantum token — leave empty to use env var or saved creds
IBM_QUANTUM_TOKEN = os.environ.get("IBM_QUANTUM_TOKEN", "")

# Build the backend object used throughout this notebook
_token_arg = IBM_QUANTUM_TOKEN if IBM_QUANTUM_TOKEN else None
backend = get_backend(BACKEND_NAME, token=_token_arg)
print(f'Backend ready: {backend}')

Backend ready: LegacyAerSimulator('aer_simulator')


In [2]:
#Amitava Chakraborty - 26th Dec 2019
# Importing standard Qiskit libraries and configuring account
import numpy as np
from qiskit import QuantumRegister, QuantumCircuit
from qiskit_backend_config import execute, Aer
from qiskit.visualization import plot_histogram, plot_bloch_multivector

import time

#IBMQ provider
from qiskit_backend_config import IBMQ

#To access IBMQ devices, we need an API token. 
MY_API_TOKEN='<enter-token-here>'

#The available backends
IBMQ.save_account(MY_API_TOKEN, overwrite=True)
provider = IBMQ.load_account()

In [3]:
providers = IBMQ.providers()

#The IBMQ provider uses a queue to allocate the devices to users. 
#We now choose a device with the least busy queue which can support our program (has at least 3 qubits).
from qiskit_backend_config import least_busy

n_qubits=4
suitableDevices = []
for provider in providers:
    print("Provider: ##:"+str(provider))
    for backend in provider.backends(filters=lambda x: x.configuration().n_qubits > n_qubits and
                                                       not x.configuration().simulator, operational=True):
        print(backend)
        cfg = backend.configuration()
        status = backend.status()
        print(str(cfg.n_qubits), status.pending_jobs, backend.job_limit())
        suitableDevices.append(backend)

        
print(suitableDevices)

backend = least_busy(suitableDevices)
ibmqc_name = backend.name
print("The best backend with "+str(n_qubits)+" qubits is " +ibmqc_name)

qiskit_runtime_service._discover_account:WARNING:2026-11-01 01:39:15,019: Loading account with the given token. A saved account will not be used.


Provider: ##:<qiskit_backend_config._ProviderShim object at 0x0000024D6AF75F70>
LegacyAerSimulator('aer_simulator')
28 0 None
[LegacyAerSimulator('aer_simulator')]
The best backend with 4 qubits is aer_simulator


h:\DevelopmentWorkspaces\gitProjects\quantumcomputing-amc\qiskit-jupyter-notebooks_dev\qiskit_backend_config.py:466: UserWarning: Could not list IBM backends ('Unable to retrieve instances. Please check that you are using a valid API token.'); using the local simulator.
  warnings.warn(f"Could not list IBM backends ({exc}); using the local simulator.")


In [4]:
#A small Bell-state circuit to run
circuit = QuantumCircuit(2, 2)
circuit.h(0)
circuit.cx(0, 1)
circuit.measure([0, 1], [0, 1])

#Execute the job in a real quantum computer
job = execute(circuit, backend=backend, shots=100)
#Print the job details and status
print (job, job.job_id(), job.status())
while job.status().name != 'DONE':
    print (job.job_id(), job.status())
    time.sleep(10)

#Get the results of the run
results = job.result()

<qiskit_backend_config.execute.<locals>._FakeJob object at 0x0000024D6AF43770> 79d0c45a-7f01-4603-b7e3-c98e6f7b260e JobStatus.RUNNING
79d0c45a-7f01-4603-b7e3-c98e6f7b260e JobStatus.RUNNING
